# PyTorch Basics: Tensors, GPU, and Autograd

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/01-PyTorch-Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_sum.py', 'q_grad_sin.py', 'q_gd.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import torch
import numpy as np
print(torch.__version__)

## Check the GPU

On Colab you should see `Tesla T4`. The T4 uses the Turing architecture (compute capability 7.5):
* It supports **fp16** Tensor Cores (fast)
* It does **not** support native bf16 or FlashAttention 2 (those need Ampere or newer)

So throughout this course we train on the T4 with fp16 mixed precision.

In [ ]:
!nvidia-smi

In [ ]:
print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(torch.cuda.get_device_name(), torch.cuda.get_device_capability())

## Tensor

A tensor is basically a numpy ndarray that can live on the GPU and supports automatic differentiation.

In [ ]:
torch.tensor(42)

In [ ]:
torch.tensor(42.)

In [ ]:
torch.tensor([42])

In [ ]:
matrix1 = torch.tensor([[3., 3.]])
matrix2 = torch.tensor([[2.], [2.]])
matrix1, matrix2

In [ ]:
product = matrix1 @ matrix2   # same as torch.matmul(matrix1, matrix2)
product

In [ ]:
# Common attributes
product.shape, product.dtype, product.device

In [ ]:
# Get the Python number
product.item()

### Converting to and from numpy

In [ ]:
a = np.arange(6).reshape(2, 3)
t = torch.from_numpy(a)   # shares memory
t

In [ ]:
t[0, 0] = 100
a

In [ ]:
t.numpy()

In [ ]:
# torch.tensor makes a copy instead
torch.tensor(a, dtype=torch.float32)

### Ways to create tensors (almost the same as numpy)

In [ ]:
torch.zeros(2, 3), torch.ones(2, 3), torch.arange(10), torch.linspace(0, 1, 5)

In [ ]:
torch.rand(2, 3), torch.randn(2, 3)

In [ ]:
x = torch.arange(12).reshape(3, 4)
x

In [ ]:
# indexing, slicing, and broadcasting work just like numpy
x[1], x[:, 1], x[x > 5], x + torch.tensor([100, 200, 300, 400])

In [ ]:
# numpy's axis is called dim in torch (most functions accept either)
x.sum(dim=0), x.float().mean(dim=1)

## Q
Compute $1+2+\cdots+10$ in different ways:
* Using the formula $10\cdot(10+1)/2$
* Using `torch.sum`
* Using `torch.cumsum`
* Using matrix multiplication `@`
* Using a Python loop

In [ ]:
# Reference solution: in Jupyter use %load q_sum.py; in Colab use !cat q_sum.py to view the code
%run -i q_sum.py

## GPU

Use `.to(device)` to move a tensor to the GPU. Tensors in the same operation must be on the same device.

In [ ]:
x = torch.randn(3, 3)
x_gpu = x.to(device)
x_gpu.device

In [ ]:
# You can also create tensors directly on the GPU
y_gpu = torch.ones(3, 3, device=device)
x_gpu + y_gpu

In [ ]:
# To convert back to numpy, move to the CPU first
(x_gpu + y_gpu).cpu().numpy()

### CPU vs GPU speed

Note that GPU operations are **asynchronous**, so call `torch.cuda.synchronize()` before and after timing.

In [ ]:
import time
def benchmark(device, dtype=torch.float32, n=4096, repeat=10):
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    a @ b  # warm up
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeat):
        c = a @ b
    if device == "cuda": torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / repeat
    print(f"{device} {dtype}: {dt*1000:.1f} ms,  {2*n**3/dt/1e12:.2f} TFLOPS")

benchmark("cpu", repeat=2)
if device == "cuda":
    benchmark("cuda", torch.float32)
    benchmark("cuda", torch.float16)   # the T4's Tensor Cores only shine in fp16

## Autograd: automatic differentiation

Set `requires_grad=True` and PyTorch records the operations (a dynamic computation graph); calling `.backward()` then computes the gradients.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1
y

In [ ]:
y.backward()
x.grad    # dy/dx = 2x + 2 = 8

In [ ]:
# Multiple variables
w = torch.tensor([1., 2., 3.], requires_grad=True)
loss = (w**2).sum()
loss.backward()
w.grad

Note: gradients **accumulate**, so clear them before each new computation.

In [ ]:
loss = (w**2).sum()
loss.backward()
w.grad   # now doubled

In [ ]:
w.grad.zero_()
w.grad

When you don't need gradients (e.g. during inference), use `torch.no_grad()` or `torch.inference_mode()` to save memory and run faster.

In [ ]:
with torch.no_grad():
    z = w * 2
z.requires_grad

In [ ]:
# .detach() returns a tensor that does not track gradients
w.detach()

## Q
Use autograd to compute the derivative of $\sin(x)$ at 100 points in $x\in[0, 2\pi]$, compare it with $\cos(x)$, and plot both.

Hint: for a vector `y`, you can use `y.sum().backward()`.

In [ ]:
# %load q_grad_sin.py
%run -i q_grad_sin.py

## Gradient descent

Once we have gradients, we can use gradient descent to find a minimum.

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
lr = 0.1
for i in range(30):
    y = (x - 2)**2
    y.backward()
    with torch.no_grad():
        x -= lr * x.grad
    x.grad.zero_()
    if i % 5 == 0:
        print(i, x.item(), y.item())

PyTorch already implements the manual parameter update above for us: that is `torch.optim`.

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
optimizer = torch.optim.SGD([x], lr=0.1)
for i in range(30):
    optimizer.zero_grad()
    y = (x - 2)**2
    y.backward()
    optimizer.step()
x

## Q
Use gradient descent to find the minimum of the Rosenbrock function
$$f(x, y) = (1-x)^2 + 100(y-x^2)^2$$
(the answer is $(1,1)$).
* Start from $(-1, 2)$
* Compare `torch.optim.SGD` and `torch.optim.Adam`
* Try different learning rates

In [ ]:
# %load q_gd.py
%run -i q_gd.py